# 03 · Milestone 1 baseline generation: 5 prompts → images

This notebook runs Stable Diffusion v1.5 (fp16, attention slicing) with fixed settings: seed 42, 30 steps, guidance 7.5, PNDM. It generates 2 real COCO captions plus 3 custom prompts of increasing complexity. Every image gets a JSON sidecar, and `prompts.json` logs the whole run.

In [ ]:
# --- Setup: mount Drive, clone or update the repo, install pinned deps. Safe to rerun. ---
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/Pranav1011/ie7615-generative-project.git"
BRANCH = "main"
IN_COLAB = "COLAB_RELEASE_TAG" in os.environ

def _git(*args):
    """Run git. For a private repo, add a GITHUB_TOKEN Colab secret (key icon in the sidebar).

    The token goes in a one-off HTTP header, so it is never written to .git/config."""
    token = None
    if IN_COLAB:
        try:
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN")
        except Exception:
            token = None
    auth = []
    if token:
        import base64
        basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
        auth = ["-c", f"http.https://github.com/.extraheader=Authorization: Basic {basic}"]
    res = subprocess.run(["git", *auth, *args], capture_output=True, text=True)
    out = res.stdout + res.stderr
    if token:
        out = out.replace(token, "***").replace(basic, "***")
    if res.returncode != 0:
        raise RuntimeError(f"git {args[0]} failed:\n{out}")
    return out

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content/ie7615-generative-project")
    if (REPO_DIR / ".git").exists():
        print(_git("-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH))
    else:
        _git("clone", "-q", "-b", BRANCH, REPO_URL, str(REPO_DIR))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")], check=True)
else:
    REPO_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print("repo:", REPO_DIR, "| commit:", subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())

In [ ]:
from src.utils import load_config, get_paths, set_seed
cfg = load_config()
paths = get_paths(cfg)
set_seed(cfg["seed"])
print("artifact root:", paths["root"])

In [ ]:
import pandas as pd
from src.generate import load_pipeline, run_prompt_set

prompts = cfg["milestone1_prompts"]
display(pd.DataFrame(prompts)[["id", "source", "complexity", "prompt"]])
pipe = load_pipeline(cfg)
print(type(pipe.scheduler).__name__, pipe.unet.dtype, pipe.device)

In [ ]:
out_dir = paths["milestone1_samples"]
images, metas, grid = run_prompt_set(pipe, prompts, cfg, out_dir)
display(grid)

### Real COCO image vs. generated image (the two COCO-caption prompts)

In [ ]:
from PIL import Image
from src import data, preprocess
from src.utils import make_image_grid

by_id = {r["image_id"]: r for r in data.load_manifest(paths["manifest"])} if paths["manifest"].exists() else {}
pairs, labels = [], []
for p, img in zip(prompts, images):
    r = by_id.get(p.get("image_id"))
    if r and preprocess.cached_path(r, paths["root"]).exists():
        pairs += [Image.open(preprocess.cached_path(r, paths["root"])), img]
        labels += [f"COCO {r['image_id']} (real)", f"{p['id']} (generated)"]
if pairs:
    cmp_grid = make_image_grid(pairs, labels, ncols=2, cell_size=320)
    cmp_grid.save(out_dir / "coco_real_vs_generated.png")
    display(cmp_grid)
else:
    print("Run notebook 01 first to see the real COCO images alongside.")

### Save: mirror to Drive and download
Put the downloaded files into `outputs/milestone1_samples/` in your local clone and commit them.

In [ ]:
import shutil
drive_copy = paths["outputs"] / "milestone1_samples"
shutil.copytree(out_dir, drive_copy, dirs_exist_ok=True)
print("mirrored to", drive_copy)
print(sorted(p.name for p in out_dir.iterdir()))
archive = shutil.make_archive("/content/milestone1_samples" if IN_COLAB else str(paths["outputs"] / "milestone1_samples"), "zip", out_dir)
if IN_COLAB:
    from google.colab import files
    files.download(archive)